# Outliers and Noise 
Outliers are extreme-but-possibly-valid points; noise is random error or garbage.

1. Outlier vs noise: a small dataset with both
2. Detection: box plot, z-score, IQR (and why z-score can be fooled)
3. Effect on statistics and on a linear model
4. Handling: remove invalid values, log transform, winsorize, robust scaling
5. Robust models: median/MAD, Huber loss regression (IRLS and gradient descent)
6. Noise: duplicates, moving-average smoothing
7. Handling strategies compared on the same test set

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(3)

## 1. Outlier vs noise
Salaries (k$) for 300 people. We inject:
* valid outliers: a few CEOs / rare high earners (real, keep them!)
* noise: impossible ages (300 yrs) and a broken sensor value (9999)

In [ ]:
n = 300
age = rng.normal(38, 9, n).clip(18, 70)
salary = 20 + 1.2 * age + rng.normal(0, 6, n)           # clean relationship
true_salary = salary.copy()
ceo = rng.choice(n, 4, replace=False); salary[ceo] += rng.uniform(150, 300, 4)   # valid outliers
bad = rng.choice(np.setdiff1d(np.arange(n), ceo), 3, replace=False)
age_bad = age.copy(); age_bad[bad] = [300, -5, 250]                               # noise: impossible ages
sensor = rng.normal(22, 1.5, n); sensor[rng.choice(n, 3, replace=False)] = 9999   # noise: broken sensor
print("CEO rows:", sorted(ceo), "| impossible-age rows:", sorted(bad))
print("Age range with noise:", age_bad.min(), age_bad.max())

## 2. Detection
**Z-score**: `z = (x - mean)/std`, flag `|z| > 3`. Mean and std are themselves
dragged by outliers, so large outliers can mask each other.
**IQR**: flag `x < Q1 - 1.5 IQR` or `x > Q3 + 1.5 IQR` (quartiles are robust).
**Modified z-score** (extra): uses median and MAD, `0.6745 (x - med)/MAD`, flag > 3.5.

In [ ]:
def zscore_flags(x, thr=3.0):
    z = (x - x.mean()) / x.std(); return np.abs(z) > thr

def quantile(x, q):
    """Linear-interpolated quantile, from scratch."""
    s = np.sort(x); pos = q * (len(s) - 1); lo = int(np.floor(pos)); hi = int(np.ceil(pos))
    return s[lo] + (s[hi] - s[lo]) * (pos - lo)

def iqr_fences(x, k=1.5):
    q1, q3 = quantile(x, .25), quantile(x, .75); i = q3 - q1
    return q1 - k * i, q3 + k * i

def iqr_flags(x, k=1.5):
    lo, hi = iqr_fences(x, k); return (x < lo) | (x > hi)

def mad(x): return np.median(np.abs(x - np.median(x)))
def modz_flags(x, thr=3.5): return np.abs(0.6745 * (x - np.median(x)) / mad(x)) > thr

# doc-style tiny example
tiny = np.array([10, 12, 11, 13, 12, 11, 10, 95.])
print("(tiny example: the outlier inflates std, so z-score misses it)\ntiny z:", zscore_flags(tiny).sum(), "| IQR:", iqr_flags(tiny).sum(), "| mod-z:", modz_flags(tiny).sum())

for nm, f in [("z-score", zscore_flags), ("IQR", iqr_flags), ("modified z", modz_flags)]:
    fl = f(salary); print(f"{nm:11s} flags {fl.sum():3d} salary points | CEOs caught: {np.isin(ceo, np.where(fl)[0]).sum()}/4")

# Masking: many extreme outliers inflate std so z-score misses them
mask_demo = np.r_[rng.normal(0, 1, 100), np.full(12, 40.0) + rng.normal(0, 1, 12)]
print("Masking demo (12 true outliers): z-score finds", zscore_flags(mask_demo).sum(),
      "| IQR finds", iqr_flags(mask_demo).sum())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].boxplot(salary, vert=False, flierprops=dict(marker='o', mfc='crimson'))
ax[0].set_title("Box plot of salary (dots = IQR outliers)"); ax[0].set_xlabel("salary (k$)")
ax[1].hist(salary, 40, color='steelblue'); lo, hi = iqr_fences(salary)
ax[1].axvline(hi, c='crimson', ls='--', label=f"IQR upper fence {hi:.0f}")
z3 = salary.mean() + 3 * salary.std(); ax[1].axvline(z3, c='orange', ls='--', label=f"z=3 at {z3:.0f}")
ax[1].set_title("Histogram with thresholds"); ax[1].legend()
fl = iqr_flags(salary); ax[2].scatter(age[~fl], salary[~fl], s=12, alpha=.6); ax[2].scatter(age[fl], salary[fl], s=40, c='crimson')
ax[2].set_title("Scatter: flagged points in red"); ax[2].set_xlabel("age"); ax[2].set_ylabel("salary")
plt.tight_layout(); plt.savefig('plots/01_detection.png', dpi=110); plt.close()

## 3. Why outliers are a problem
Mean/std and least-squares regression are pulled toward extremes. Compare the
statistics and the OLS fit with and without the 4 CEO rows.

In [ ]:
def ols(x, y):
    A = np.c_[np.ones(len(x)), x]; return np.linalg.lstsq(A, y, rcond=None)[0]

keep = np.ones(n, bool); keep[ceo] = False
print(f"mean  all {salary.mean():.1f} | without CEOs {salary[keep].mean():.1f} | median {np.median(salary):.1f}")
print(f"std   all {salary.std():.1f} | without CEOs {salary[keep].std():.1f} | MAD-std {1.4826*mad(salary):.1f}")
w_all = ols(age, salary); w_clean = ols(age[keep], salary[keep])
print("OLS (intercept, slope) all:", np.round(w_all, 2), "| w/o CEOs:", np.round(w_clean, 2), "| truth: [20, 1.2]")

## 4. Handling
* **Remove invalid values** (age 300): domain rules, not statistics
* **Log transform**: tames right skew
* **Winsorize**: cap at percentiles (keeps the row)
* **Robust scaling**: `(x - median)/IQR`, unaffected by extremes

In [ ]:
def valid_age_mask(a): return (a >= 0) & (a <= 120)
print("Rows removed by domain rule (0<=age<=120):", (~valid_age_mask(age_bad)).sum())

def winsorize(x, lo_q=0.05, hi_q=0.95):
    lo, hi = quantile(x, lo_q), quantile(x, hi_q); return np.clip(x, lo, hi)

def standard_scale(x): return (x - x.mean()) / x.std()
def robust_scale(x): return (x - np.median(x)) / (quantile(x, .75) - quantile(x, .25))
def minmax_scale(x): return (x - x.min()) / (x.max() - x.min())

# sensor with 9999 spikes: how each scaler behaves on the normal readings
normal_mask = sensor < 100
for nm, f in [("min-max", minmax_scale), ("standard", standard_scale), ("robust", robust_scale)]:
    s = f(sensor); print(f"{nm:9s}: spread (std) of the 297 normal readings after scaling = {s[normal_mask].std():.4f}")

skewed = np.exp(rng.normal(3, 1, 500)); skewed[:3] *= 40
def skew(x): z = (x - x.mean()) / x.std(); return float(np.mean(z ** 3))
print(f"skewness raw {skew(skewed):.2f} -> log {skew(np.log(skewed)):.2f} -> sqrt {skew(np.sqrt(skewed)):.2f}")

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
ax[0, 0].hist(skewed, 50); ax[0, 0].set_title("Skewed salary-like data")
ax[0, 1].hist(np.log(skewed), 50, color='g'); ax[0, 1].set_title("After log transform")
ax[0, 2].hist(winsorize(skewed), 50, color='purple'); ax[0, 2].set_title("Winsorized 5%-95%")
for a, (nm, f) in zip(ax[1], [("min-max", minmax_scale), ("standard", standard_scale), ("robust", robust_scale)]):
    a.hist(f(sensor)[normal_mask], 30); a.set_title(f"Sensor after {nm} scaling\n(normal readings only)")
plt.tight_layout(); plt.savefig('plots/02_handling.png', dpi=110); plt.close()

## 5. Robust models
**Huber loss**: quadratic for small residuals, linear for large ones
`L(r) = r^2/2 if |r|<=d else d(|r| - d/2)`. Gradient is clipped at `d`, so one huge
residual can't dominate. Fit with IRLS (weights `w = min(1, d/|r|)`) and verify
against plain gradient descent on the same loss.

In [ ]:
def huber_loss(r, d=1.345):
    a = np.abs(r); return np.where(a <= d, 0.5 * r ** 2, d * (a - 0.5 * d))

def huber_irls(x, y, iters=50):
    A = np.c_[np.ones(len(x)), x]; w = np.linalg.lstsq(A, y, rcond=None)[0]
    for _ in range(iters):
        r = y - A @ w; s = 1.4826 * mad(r) + 1e-12; d = 1.345 * s    # scale from MAD
        wt = np.minimum(1, d / np.maximum(np.abs(r), 1e-12))
        W = A.T * wt; w = np.linalg.solve(W @ A, W @ y)
    return w

def huber_gd(x, y, d=5.0, lr=0.05, epochs=4000):
    mu, sd = x.mean(), x.std(); z = (x - mu) / sd; b0, b1 = np.median(y), 0.0
    for _ in range(epochs):
        r = y - (b0 + b1 * z); g = np.clip(r, -d, d)          # d/dr of Huber, clipped residual
        b0 += lr * g.mean(); b1 += lr * (g * z).mean()
    return np.array([b0 - b1 * mu / sd, b1 / sd])

def lad_irls(x, y, iters=100):
    A = np.c_[np.ones(len(x)), x]; w = np.linalg.lstsq(A, y, rcond=None)[0]
    for _ in range(iters):
        wt = 1 / np.maximum(np.abs(y - A @ w), 1e-4); W = A.T * wt; w = np.linalg.solve(W @ A, W @ y)
    return w

w_h = huber_irls(age, salary); w_g = huber_gd(age, salary); w_l = lad_irls(age, salary)
print("OLS (all rows):", np.round(w_all, 2), "| Huber IRLS:", np.round(w_h, 2),
      "| Huber GD:", np.round(w_g, 2), "| LAD:", np.round(w_l, 2), "| truth: [20, 1.2]")

In [ ]:
xs = np.linspace(18, 70, 100); fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].scatter(age, salary, s=12, alpha=.5); ax[0].scatter(age[ceo], salary[ceo], c='crimson', s=40, label='CEOs')
for nm, w, c in [("OLS", w_all, 'orange'), ("Huber", w_h, 'g'), ("LAD", w_l, 'purple'), ("truth", np.array([20, 1.2]), 'k')]:
    ax[0].plot(xs, w[0] + w[1] * xs, c=c, label=nm, ls='--' if nm == 'truth' else '-')
ax[0].legend(); ax[0].set_title("Regression with 4 valid outliers"); ax[0].set_xlabel("age"); ax[0].set_ylabel("salary")
r = np.linspace(-8, 8, 200); ax[1].plot(r, 0.5 * r ** 2, label="squared / 2"); ax[1].plot(r, np.abs(r), label="absolute")
ax[1].plot(r, huber_loss(r, 2.0), label="Huber (d=2)"); ax[1].set_ylim(0, 20); ax[1].legend(); ax[1].set_title("Loss functions")
plt.tight_layout(); plt.savefig('plots/03_robust_regression.png', dpi=110); plt.close()

Robust statistics: how do mean/median/std/MAD move as one point goes to infinity?

In [ ]:
base = rng.normal(50, 5, 101); vals = np.array([0, 10, 100, 1e3, 1e4]); rows = []
for v in vals:
    x = base.copy(); x[0] = v; rows.append([x.mean(), np.median(x), x.std(), 1.4826 * mad(x)])
rows = np.array(rows); print("outlier   mean   median    std   MAD-std")
for v, rw in zip(vals, rows): print(f"{v:7.0f} {rw[0]:7.1f} {rw[1]:7.1f} {rw[2]:8.1f} {rw[3]:7.1f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogx(vals + 1, rows[:, 0], 'o-', label='mean'); ax[0].semilogx(vals + 1, rows[:, 1], 's-', label='median'); ax[0].legend(); ax[0].set_title("Location vs outlier size")
ax[1].loglog(vals + 1, rows[:, 2], 'o-', label='std'); ax[1].loglog(vals + 1, rows[:, 3], 's-', label='MAD-std'); ax[1].legend(); ax[1].set_title("Spread vs outlier size")
plt.tight_layout(); plt.savefig('plots/04_robust_statistics.png', dpi=110); plt.close()

## 6. Noise: duplicates and smoothing
Remove duplicate records; smooth a noisy signal with a moving average and a
median filter (the median filter kills spikes, the average only smears them).

In [ ]:
def dedupe(rows):
    _, idx = np.unique(rows, axis=0, return_index=True); return rows[np.sort(idx)]

recs = np.c_[rng.integers(0, 50, 80), rng.integers(0, 5, 80)]; recs = np.vstack([recs, recs[:15]])
print("records before/after dedupe:", len(recs), len(dedupe(recs)))

def moving_average(x, k=9):
    pad = k // 2; xp = np.r_[np.full(pad, x[0]), x, np.full(pad, x[-1])]
    return np.convolve(xp, np.ones(k) / k, mode='valid')

def median_filter(x, k=9):
    pad = k // 2; xp = np.r_[np.full(pad, x[0]), x, np.full(pad, x[-1])]
    return np.array([np.median(xp[i:i + k]) for i in range(len(x))])

t = np.linspace(0, 4 * np.pi, 300); clean_sig = np.sin(t)
noisy = clean_sig + rng.normal(0, 0.3, 300); spikes = rng.choice(300, 8, replace=False); noisy[spikes] += rng.choice([-5, 5], 8)
ma, mf = moving_average(noisy), median_filter(noisy)
mse = lambda a: float(np.mean((a - clean_sig) ** 2))
print(f"MSE vs clean signal: noisy {mse(noisy):.3f} | moving avg {mse(ma):.3f} | median filter {mse(mf):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for a, (nm, s) in zip(ax, [("Noisy + spikes", noisy), ("Moving average (k=9)", ma), ("Median filter (k=9)", mf)]):
    a.plot(t, s, lw=1); a.plot(t, clean_sig, 'k--', lw=1); a.set_title(nm)
plt.tight_layout(); plt.savefig('plots/05_noise_smoothing.png', dpi=110); plt.close()

## 7. Strategies compared
Train on data containing outliers and noise, test on clean held-out data.
Training set: 4% valid high-salary outliers and 4% garbage y values (noise).
Strategies: do nothing, IQR removal, winsorize y, log target, Huber, LAD.
Repeated over 30 seeds. Note the difference between *noise* (should clean) and
*valid outliers* (removal here can hurt if the test set also has them).

In [ ]:
def experiment(seed, p_valid=0.04, p_noise=0.04):
    g = np.random.default_rng(seed); m = 400
    a = g.normal(38, 9, m); y = 20 + 1.2 * a + g.normal(0, 6, m)
    atr, ytr_clean, ate = a[:300], y[:300], a[300:]; yte = y[300:]
    ytr = ytr_clean.copy(); k1 = g.choice(300, int(300 * p_valid), replace=False); ytr[k1] += g.uniform(100, 250, len(k1))
    k2 = g.choice(300, int(300 * p_noise), replace=False); ytr[k2] = g.choice([-500, 5000], len(k2))
    res = {}; ev = lambda w: float(np.sqrt(np.mean((w[0] + w[1] * ate - yte) ** 2)))
    res["raw OLS"] = ev(ols(atr, ytr))
    kk = ~iqr_flags(ytr); res["IQR remove"] = ev(ols(atr[kk], ytr[kk]))
    res["winsorize"] = ev(ols(atr, winsorize(ytr, .05, .95)))
    yl = np.log(np.maximum(ytr, 1)); wl = ols(atr, yl); res["log target"] = float(np.sqrt(np.mean((np.exp(wl[0] + wl[1] * ate) - yte) ** 2)))
    res["Huber"] = ev(huber_irls(atr, ytr)); res["LAD"] = ev(lad_irls(atr, ytr))
    return res

allr = [experiment(s) for s in range(30)]; names = list(allr[0])
avg = {k: np.mean([r[k] for r in allr]) for k in names}; sdv = {k: np.std([r[k] for r in allr]) for k in names}
for k in names: print(f"{k:12s} test RMSE {avg[k]:7.2f} +/- {sdv[k]:.2f}")
fig, ax = plt.subplots(figsize=(8, 4)); ax.bar(names, [avg[k] for k in names], yerr=[sdv[k] for k in names], capsize=3)
ax.set_yscale('log'); ax.set_ylabel("test RMSE (log scale)"); ax.set_title("Strategies on contaminated training data")
plt.xticks(rotation=20); plt.tight_layout(); plt.savefig('plots/06_strategy_comparison.png', dpi=110); plt.close()

**Golden rule**: investigate first. Remove errors (age 300), keep or down-weight valid
extremes (CEO salary, fraud), and prefer robust tools when you cannot tell the difference.

In [ ]:
# Optional sanity check against numpy (not sklearn) for the from-scratch pieces
print("quantile matches numpy:", np.isclose(quantile(salary, .25), np.quantile(salary, .25)),
      np.isclose(quantile(salary, .75), np.quantile(salary, .75)))
try:
    from sklearn.preprocessing import RobustScaler
    rs = RobustScaler().fit_transform(salary[:, None])[:, 0]
    print("robust_scale matches sklearn RobustScaler:", np.allclose(rs, robust_scale(salary)))
except Exception as e:
    print("sklearn check skipped:", e)